In [1]:
# Importing Modules
import torch
import pandas as pd
import numpy as np
import itertools
from src.dataloader import loadData
from src.model import SimpleGCN, SimpleGAT, SimpleGIN, SimpleGraphSAGE
from src.train_test import TrainGNN, TestGNN
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
######################## DATA-1 ##################################
# Loading ESOL data
esol_train_data = pd.read_csv("data/train/ESOL.csv")
esol_test_data = pd.read_csv("data/test/ESOL.csv")

######################## DATA-2 ##################################
# Loading FreeSolv data
freeSolv_train_data = pd.read_csv("data/train/FreeSolv.csv")
freeSolv_test_data = pd.read_csv("data/test/FreeSolv.csv")

######################## DATA-3 ##################################
# Loading Lipophilicity data
lipophilicity_train_data = pd.read_csv("data/train/Lipophilicity.csv")
lipophilicity_test_data = pd.read_csv("data/test/Lipophilicity.csv")

In [3]:
# Models
model_dict = {
        "SimpleGCN":SimpleGAT, 
		"SimpleGAT":SimpleGAT, 
		"SimpleGIN":SimpleGIN,
		"SimpleGraphSAGE":SimpleGraphSAGE
}

# Models Hyperparameters
model_hyperparams_dict = {
        "SimpleGCN":[0.001, 128, 4],
        "SimpleGAT":[0.005, 32, 8],
        "SimpleGIN":[0.005, 128, 8],
        "SimpleGraphSAGE":[0.001, 128, 4]
}

# Function to run analysis
def RunGNN(model, train_data, test_data, dataName, modelName, params):
    
	# Training data and target labels
	smiles_X_train = train_data["smiles"]
	X_train = smiles_X_train.to_numpy()
	y_train = train_data["target"]
	y_train = y_train.to_numpy()

    # Testing data and target labels
	smiles_X_test = test_data["smiles"]
	X_test = smiles_X_test.to_numpy()
	y_test = test_data["target"]
	y_test = y_test.to_numpy()

    # Params
	lr, h_dim, b_size = params

	# Loading dataset
	train_loader = loadData(X_train, y_train, batch_size=b_size)
	test_loader = loadData(X_test, y_test, batch_size=b_size)

	# Model
	model = model_dict[modelName](num_features=11, hidden_dim=h_dim, num_classes=1)

	# Model training
	model = TrainGNN(model, train_loader, epochs=100, learning_rate=lr)

	# Model testing
	y_test, y_pred = TestGNN(model, test_loader)

	# Calculate MAE
	mae = mean_absolute_error(y_test, y_pred)

	# Calculate RMSE
	rmse = root_mean_squared_error(y_test, y_pred)
    
    # Calculate pearson r and p_val
	r, p_val = pearsonr(y_test, y_pred)
    
	# Return results
	return pd.DataFrame({"Data":[dataName], "Model":[modelName],
			"MAE":round(mae, 2), "RMSE":[round(rmse, 2)], 
			"r":[round(r, 2)], "p-val":[round(p_val, 3)]})

In [4]:
# Data dict
datasets = {
    "ESOL":{"train":esol_train_data, "test":esol_test_data},
    "FreeSolv":{"train":freeSolv_train_data, "test":freeSolv_test_data},
    "Lipophilicity":{"train":lipophilicity_train_data, "test":lipophilicity_test_data}
}

In [5]:
# List to store results
temp_out = []

# Loop for models
for modelName, model in model_dict.items():
    # Loop for dataset
    for dataName, data in datasets.items():
        # Run Analysis for model and dataset
        temp_out.append(RunGNN(model, data["train"], data["test"], dataName, modelName, model_hyperparams_dict[modelName]))

# Final output
GNN_out = pd.concat(temp_out, ignore_index=True)
GNN_out.to_csv("results/Output_GNN_Analysis.csv")
GNN_out

,Data,Model,MAE,RMSE,r,p-val
0,ESOL,SimpleGCN,1.08,1.34,0.81,0.0
1,FreeSolv,SimpleGCN,2.60,3.70,0.62,0.0
2,Lipophilicity,SimpleGCN,0.86,1.07,0.56,0.0
3,ESOL,SimpleGAT,1.11,1.39,0.78,0.0
4,FreeSolv,SimpleGAT,2.55,3.58,0.72,0.0
5,Lipophilicity,SimpleGAT,0.82,1.03,0.56,0.0
6,ESOL,SimpleGIN,1.07,1.33,0.79,0.0
7,FreeSolv,SimpleGIN,2.79,3.34,0.73,0.0
8,Lipophilicity,SimpleGIN,0.95,1.15,0.35,0.0
9,ESOL,SimpleGraphSAGE,1.00,1.31,0.81,0.0


In [6]:
# Function to plot barplot showing result
def plot_bar(data, target):
    sns.set_theme(style="ticks", context="paper")
    g = sns.catplot(data=data, kind="bar", x="Model", y=target, hue="Model",
                    col="Data", col_wrap=3, sharey=False, height=4, width=0.8,
                    aspect=0.8, dodge=False)
    g.set_axis_labels("Model", f"{target}")
    g.set_titles("{col_name}")
    for ax in g.axes.flat:
        ax.tick_params(axis="x", rotation=40)
    plt.tight_layout()
    plt.savefig(f"results/GNN_Analysis_{target}_Plot.png", dpi=300)
    plt.close()

In [7]:
# Plotting barplot for RMSE
plot_bar(GNN_out, "RMSE")
# Plotting barplot for MAE
plot_bar(GNN_out, "MAE")
# Plotting barplot for 
plot_bar(GNN_out, "r")